# lltRewardModels: best-of-n pool, gold scores and the proxy reward models

One T4 session. Writes llt_rm_main.json, pool.json, gold and proxy score files and three fp16 checkpoints.

In [ ]:
TAG = 'main'

import json, os, subprocess, sys, time, platform, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert sys.version_info >= (3, 11), "Python 3.11 or newer is needed"
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi not found: this session has no GPU")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "trl==1.17.1", "peft==0.21.2", "accelerate==1.15.0",
                "datasets==5.1.0"], check=False)
WORKER = '"""lltRewardModels Run Library worker. One job per process.\n\nUsage: python worker_rm.py <job.json> <result.json>\njob.kind:\n  sample     best-of-n pool: n completions per prompt from SmolLM2-360M-Instruct (fp32 sampling)\n  gold       score the pool, held-out UltraFeedback pairs and a RewardBench 2 subset with the gold RM\n  train      Bradley-Terry reward model, written by hand, SmolLM2-360M + scalar head, full fine-tune\n  trlcheck   own loss vs TRL RewardTrainer.compute_loss on the same batch, then a short RewardTrainer run\nAll data selection is deterministic (see select_pairs, select_prompts, rb2_subset).\n"""\nimport gc, json, math, os, random, sys, time, traceback\nos.environ["CUDA_VISIBLE_DEVICES"] = "0"\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\njob = json.load(open(sys.argv[1])); OUT = sys.argv[2]\nR = {"tag": job["tag"], "kind": job["kind"], "job": job}\nWORK = job.get("work", "/kaggle/working")\n\n\ndef save():\n    json.dump(R, open(OUT, "w"), indent=1, default=str)\n\n\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer\n\nDEV = "cuda:0" if torch.cuda.is_available() else "cpu"\nBASE = "HuggingFaceTB/SmolLM2-360M"\nPOLICY = "HuggingFaceTB/SmolLM2-360M-Instruct"\nGOLD = "Skywork/Skywork-Reward-V2-Qwen3-1.7B"\nUF = "HuggingFaceH4/ultrafeedback_binarized"\nMAXLEN = 1024\n\n\ndef peak():\n    return round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None\n\n\nCUDA = torch.cuda.is_available()\n\n\ndef reset_peak():\n    if CUDA:\n        torch.cuda.reset_peak_memory_stats()\n\n\ndef sync():\n    if CUDA:\n        torch.cuda.synchronize()\n\n\ndef free():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef rm_tokenizer():\n    """SmolLM2 base ships no chat template: borrow the instruct one, keep pad apart from the turn end."""\n    tok = AutoTokenizer.from_pretrained(BASE)\n    inst = AutoTokenizer.from_pretrained(POLICY)\n    tok.chat_template = inst.chat_template\n    tok.pad_token = "<|endoftext|>"\n    return tok\n\n\ndef conv(prompt, response):\n    return [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]\n\n\ndef ids_of(tok, prompt, response):\n    return list(tok.apply_chat_template(conv(prompt, response))["input_ids"])\n\n\ndef select_pairs(tok, split, n, seed=0, balanced=False, start=0):\n    """Shuffle the split with a fixed seed, keep pairs whose two sequences fit MAXLEN tokens, take n.\n    balanced=True takes n/2 pairs whose chosen is longer and n/2 whose chosen is shorter (equal lengths skipped)."""\n    from datasets import load_dataset\n    ds = load_dataset(UF, split=split)\n    if seed is not None:\n        ds = ds.shuffle(seed=seed)\n    out, longer, shorter, read = [], 0, 0, 0\n    for i in range(start, len(ds)):\n        ex = ds[i]; read += 1\n        p = ex["prompt"]; c = ex["chosen"][-1]["content"]; r = ex["rejected"][-1]["content"]\n        if c == r:\n            continue\n        ci = ids_of(tok, p, c); ri = ids_of(tok, p, r)\n        if len(ci) > MAXLEN or len(ri) > MAXLEN:\n            continue\n        row = dict(prompt=p, c=ci, r=ri, sc=ex["score_chosen"], sr=ex["score_rejected"], pid=ex["prompt_id"])\n        if balanced:\n            if len(ci) == len(ri):\n                continue\n            if len(ci) > len(ri):\n                if longer >= n // 2:\n                    continue\n                longer += 1\n            else:\n                if shorter >= n // 2:\n                    continue\n                shorter += 1\n        out.append(row)\n        if len(out) >= n:\n            break\n    return out, read\n\n\ndef select_prompts(n, max_prompt_tokens=200, max_ref_tokens=160):\n    """First n test_prefs prompts (file order) whose prompt fits max_prompt_tokens and whose chosen reference answer\n    fits max_ref_tokens, so most samples can finish inside the generation budget."""\n    from datasets import load_dataset\n    tok = AutoTokenizer.from_pretrained(POLICY)\n    ds = load_dataset(UF, split="test_prefs")\n    out = []\n    for i, ex in enumerate(ds):\n        m = [{"role": "user", "content": ex["prompt"]}]\n        L = len(tok.apply_chat_template(m, add_generation_prompt=True)["input_ids"])\n        ref = len(tok(ex["chosen"][-1]["content"], add_special_tokens=False).input_ids)\n        if L <= max_prompt_tokens and ref <= max_ref_tokens:\n            out.append(dict(row=i, prompt=ex["prompt"], prompt_id=ex["prompt_id"], promptTokens=L, refTokens=ref))\n        if len(out) >= n:\n            break\n    return out\n\n\ndef rb2_subset(per_subset, seed=0):\n    from datasets import load_dataset\n    ds = load_dataset("allenai/reward-bench-2", split="test")\n    rng = random.Random(seed)\n    out = []\n    for sub in ["Factuality", "Precise IF", "Math", "Safety", "Focus"]:\n        idx = [i for i, s in enumerate(ds["subset"]) if s == sub]\n        pick = sorted(rng.sample(idx, min(per_subset, len(idx))))\n        for i in pick:\n            ex = ds[i]\n            out.append(dict(id=ex["id"], subset=sub, prompt=ex["prompt"], chosen=list(ex["chosen"]), rejected=list(ex["rejected"])))\n    return out\n\n\ndef score_seqs(model, seqs, pad_id, max_tokens=16384, max_batch=32, dtype=torch.float16):\n    """Score token-id lists; returns rewards in the input order. Batches sorted by length, right padding."""\n    order = sorted(range(len(seqs)), key=lambda i: -len(seqs[i]))\n    out = [None] * len(seqs); i = 0\n    while i < len(order):\n        L = len(seqs[order[i]])\n        b = max(1, min(max_batch, max_tokens // max(L, 1)))\n        idx = order[i:i + b]\n        ids = torch.full((len(idx), L), pad_id, dtype=torch.long); att = torch.zeros_like(ids)\n        for k, j in enumerate(idx):\n            s = seqs[j]; ids[k, :len(s)] = torch.tensor(s); att[k, :len(s)] = 1\n        with torch.no_grad(), torch.autocast("cuda", dtype=dtype, enabled=torch.cuda.is_available()):\n            r = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV)).logits.squeeze(-1).float().cpu().tolist()\n        for k, j in enumerate(idx):\n            out[j] = r[k]\n        i += b\n    return out\n\n\ndef pair_metrics(rc, rr, pairs):\n    rc = np.array(rc); rr = np.array(rr)\n    win = (rc > rr).astype(float)\n    lc = np.array([len(p["c"]) for p in pairs]); lr = np.array([len(p["r"]) for p in pairs])\n    tie = np.array([p["sc"] == p["sr"] for p in pairs])\n    m = {"n": len(pairs), "acc": float(win.mean()), "accNoScoreTie": float(win[~tie].mean()), "nScoreTie": int(tie.sum()),\n         "accChosenLonger": float(win[lc > lr].mean()), "nChosenLonger": int((lc > lr).sum()),\n         "accChosenShorter": float(win[lc < lr].mean()), "nChosenShorter": int((lc < lr).sum()),\n         "lengthOnlyAcc": float((lc > lr).mean()), "meanMargin": float((rc - rr).mean())}\n    allr = np.concatenate([rc, rr]); alll = np.concatenate([lc, lr])\n    m["pearsonRewardLength"] = float(np.corrcoef(allr, alll)[0, 1])\n    return m, win.tolist()\n\n\ndef load_pool():\n    p = job.get("pool") or os.path.join(WORK, "pool.json")\n    return json.load(open(p))\n\n\ntry:\n    kind = job["kind"]\n    R["env"] = {"torch": torch.__version__, "transformers": transformers.__version__,\n                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\n    if kind == "sample":\n        tok = AutoTokenizer.from_pretrained(POLICY); tok.padding_side = "left"\n        prompts = select_prompts(job["prompts"], max_ref_tokens=job.get("maxRefTokens", 160))\n        t0 = time.time()\n        model = AutoModelForCausalLM.from_pretrained(POLICY, dtype=torch.float32, device_map=DEV).eval()\n        R["loadSec"] = round(time.time() - t0, 1)\n        n = job["n"]; per = job["promptsPerBatch"]; new = job["newTokens"]\n        torch.manual_seed(job.get("seed", 0))\n        pool = {"policy": POLICY, "dtype": "float32", "n": n, "newTokens": new, "temperature": 1.0, "top_p": 1.0, "top_k": 0,\n                "prompts": prompts, "samples": []}\n        im_end = tok.convert_tokens_to_ids("<|im_end|>")\n        reset_peak(); times = []\n        for b in range(0, len(prompts), per):\n            group = prompts[b:b + per]\n            msgs = [tok.apply_chat_template([{"role": "user", "content": g["prompt"]}], tokenize=False, add_generation_prompt=True)\n                    for g in group for _ in range(n)]\n            enc = tok(msgs, return_tensors="pt", padding=True).to(DEV)\n            sync(); t = time.time()\n            with torch.no_grad():\n                out = model.generate(**enc, max_new_tokens=new, do_sample=True, temperature=1.0, top_p=1.0, top_k=0,\n                                     pad_token_id=tok.pad_token_id, eos_token_id=im_end)\n            sync(); times.append(time.time() - t)\n            gen = out[:, enc.input_ids.shape[1]:].cpu()\n            for k in range(gen.shape[0]):\n                g = gen[k].tolist()\n                ended = im_end in g\n                if ended:\n                    g = g[:g.index(im_end)]\n                while g and g[-1] == tok.pad_token_id:\n                    g = g[:-1]\n                pool["samples"].append(dict(p=b + k // n, j=k % n, text=tok.decode(g, skip_special_tokens=True),\n                                            newTokens=len(g), ended=ended))\n            print("batch", b // per, "sec", round(times[-1], 1), "peak", peak(), flush=True)\n        json.dump(pool, open(os.path.join(WORK, "pool.json"), "w"))\n        ends = [s["ended"] for s in pool["samples"]]\n        R.update(ok=True, prompts=len(prompts), completions=len(pool["samples"]), batchSec=[round(x, 1) for x in times],\n                 totalGenSec=round(sum(times), 1), peakMemGB=peak(), endedShare=float(np.mean(ends)),\n                 meanNewTokens=float(np.mean([s["newTokens"] for s in pool["samples"]])))\n\n    elif kind == "gold":\n        t0 = time.time()\n        GOLD = job.get("goldModel", GOLD)\n        gtok = AutoTokenizer.from_pretrained(GOLD)\n        gold = AutoModelForSequenceClassification.from_pretrained(GOLD, num_labels=1, dtype=torch.float16, device_map=DEV).eval()\n        if gtok.pad_token is None:\n            gtok.pad_token = gtok.eos_token\n        gold.config.pad_token_id = gtok.pad_token_id\n        gold.config.get_text_config().pad_token_id = gtok.pad_token_id\n        R["loadSec"] = round(time.time() - t0, 1); R["residentGB"] = round(torch.cuda.memory_allocated() / 1e9, 2) if CUDA else None\n        R["padToken"] = gtok.pad_token; R["bos"] = gtok.bos_token\n\n        def gids(prompt, response, cap=job.get("goldMaxTokens", 4096)):\n            s = gtok.apply_chat_template(conv(prompt, response), tokenize=False)\n            if gtok.bos_token is not None and s.startswith(gtok.bos_token):\n                s = s[len(gtok.bos_token):]\n            ids = gtok(s, add_special_tokens=False).input_ids\n            return ids[:cap], len(ids) > cap\n\n        R["exampleRendered"] = gtok.apply_chat_template(conv("Name a fruit.", "A mango."), tokenize=False)\n        reset_peak()\n        # 1. the best-of-n pool\n        pool = load_pool()\n        seqs = [gids(pool["prompts"][s["p"]]["prompt"], s["text"])[0] for s in pool["samples"]]\n        t = time.time(); g = score_seqs(gold, seqs, gtok.pad_token_id, max_tokens=job.get("goldTokBudget", 12288)); R["poolSec"] = round(time.time() - t, 1)\n        json.dump({"gold": GOLD, "scores": g}, open(os.path.join(WORK, "gold_pool.json"), "w"))\n        R["poolSeqPerSec"] = round(len(seqs) / R["poolSec"], 2)\n        # 2. held-out UltraFeedback pairs (same selection as the proxy\'s held-out set), chat format\n        rtok = rm_tokenizer()\n        held, read = select_pairs(rtok, "test_prefs", job["heldPairs"], seed=None)\n        cs, rs = [], []; capped = 0\n        from datasets import load_dataset\n        te = load_dataset(UF, split="test_prefs"); byid = {}\n        for ex in te:\n            byid.setdefault(ex["prompt_id"], ex)\n        for h in held:\n            ex = byid[h["pid"]]\n            a, ca = gids(ex["prompt"], ex["chosen"][-1]["content"]); b, cb = gids(ex["prompt"], ex["rejected"][-1]["content"])\n            cs.append(a); rs.append(b); capped += int(ca) + int(cb)\n        t = time.time(); sc = score_seqs(gold, cs + rs, gtok.pad_token_id, max_tokens=job.get("goldTokBudget", 12288)); R["heldSec"] = round(time.time() - t, 1)\n        m, win = pair_metrics(sc[:len(held)], sc[len(held):], held)\n        R["heldOut"] = m; R["heldOutCapped"] = capped\n        json.dump({"gold": GOLD, "pid": [h["pid"] for h in held], "rc": sc[:len(held)], "rr": sc[len(held):], "win": win},\n                  open(os.path.join(WORK, "gold_held.json"), "w"))\n        # 3. RewardBench 2 subset, chosen must beat all three rejected\n        items = rb2_subset(job["rb2PerSubset"])\n        flat, owner = [], []\n        for k, it in enumerate(items):\n            for resp in it["chosen"][:1] + it["rejected"][:3]:\n                flat.append(gids(it["prompt"], resp, cap=job.get("rb2MaxTokens", 2048))[0]); owner.append(k)\n        t = time.time(); sc = score_seqs(gold, flat, gtok.pad_token_id, max_tokens=job.get("goldTokBudget", 12288)); R["rb2Sec"] = round(time.time() - t, 1)\n        res = {}\n        for k, it in enumerate(items):\n            s = sc[4 * k:4 * k + 4]\n            res.setdefault(it["subset"], []).append(float(s[0] > max(s[1:])))\n        R["rb2"] = {sub: {"n": len(v), "acc": float(np.mean(v))} for sub, v in res.items()}\n        R["rb2Mean"] = float(np.mean([v["acc"] for v in R["rb2"].values()]))\n        json.dump({"gold": GOLD, "ids": [it["id"] for it in items], "subset": [it["subset"] for it in items], "scores": sc},\n                  open(os.path.join(WORK, "gold_rb2.json"), "w"))\n        R.update(ok=True, peakMemGB=peak(), totalSec=round(time.time() - t0, 1))\n\n    elif kind == "train":\n        seed = job["seed"]; tag = job["tag"]\n        tok = rm_tokenizer()\n        torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)\n        t0 = time.time()\n        train, read = select_pairs(tok, "train_prefs", job["pairs"], seed=0, balanced=job.get("balanced", False))\n        held, _ = select_pairs(tok, "test_prefs", job["heldPairs"], seed=None)\n        R["data"] = {"trainPairs": len(train), "rowsRead": read, "heldPairs": len(held), "selectSec": round(time.time() - t0, 1),\n                     "trainChosenLongerShare": float(np.mean([len(p["c"]) > len(p["r"]) for p in train])),\n                     "trainScoreTieShare": float(np.mean([p["sc"] == p["sr"] for p in train])),\n                     "trainMeanTokens": float(np.mean([len(p["c"]) + len(p["r"]) for p in train]) / 2)}\n        print(R["data"], flush=True)\n        model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1, dtype=torch.float32, device_map=DEV)\n        model.config.pad_token_id = tok.pad_token_id\n        model.config.get_text_config().pad_token_id = tok.pad_token_id\n        R["head"] = {"name": "score", "shape": list(model.score.weight.shape), "bias": model.score.bias is not None,\n                     "initStd": float(model.score.weight.std())}\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.config.use_cache = False\n        params = [p for p in model.parameters() if p.requires_grad]\n        R["trainableParams"] = sum(p.numel() for p in params)\n        opt = torch.optim.AdamW(params, lr=job["lr"], weight_decay=0.0)\n        B = job["batchPairs"]; micro = job.get("microPairs", B)\n        # bucketing: shuffle, sort inside chunks of 64 pairs by length, cut into batches, shuffle the batch order\n        rng = random.Random(seed)\n        order = list(range(len(train))); rng.shuffle(order)\n        batches = []\n        for c0 in range(0, len(order), 64):\n            chunk = sorted(order[c0:c0 + 64], key=lambda i: -max(len(train[i]["c"]), len(train[i]["r"])))\n            batches += [chunk[k:k + B] for k in range(0, len(chunk), B)]\n        rng.shuffle(batches)\n        steps = len(batches); warm = max(1, int(0.03 * steps))\n        def lr_factor(s):   # linear warmup over 3% of the steps, then cosine to zero\n            if s < warm:\n                return (s + 1) / warm\n            return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, steps - warm))))\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)\n        scaler = torch.amp.GradScaler("cuda", init_scale=2048, enabled=CUDA)\n        evals_at = sorted(set(int(round(f * steps)) for f in job.get("evalFractions", [1.0])))\n        pool = load_pool() if job.get("scorePool") else None\n        pool_seqs = None\n        if pool:\n            pool_seqs = [ids_of(tok, pool["prompts"][s["p"]]["prompt"], s["text"]) for s in pool["samples"]]\n            R["poolSeqs"] = len(pool_seqs)\n        metrics, evals = [], []\n\n        def evaluate(step):\n            model.eval(); t = time.time()\n            sc = score_seqs(model, [h["c"] for h in held] + [h["r"] for h in held], tok.pad_token_id)\n            m, win = pair_metrics(sc[:len(held)], sc[len(held):], held)\n            e = {"step": step, "pairsSeen": min(step * B, len(train)), "heldOut": m, "evalSec": round(time.time() - t, 1)}\n            json.dump({"pid": [h["pid"] for h in held], "rc": sc[:len(held)], "rr": sc[len(held):], "win": win},\n                      open(os.path.join(WORK, "proxy_held_%s_step%d.json" % (tag, step)), "w"))\n            if pool_seqs is not None:\n                t2 = time.time(); ps = score_seqs(model, pool_seqs, tok.pad_token_id)\n                json.dump({"proxy": tag, "step": step, "scores": ps}, open(os.path.join(WORK, "proxy_pool_%s_step%d.json" % (tag, step)), "w"))\n                e["poolSec"] = round(time.time() - t2, 1)\n            evals.append(e); R["evals"] = evals; save()\n            print("EVAL", json.dumps(e)[:400], flush=True)\n            model.train()\n\n        if job.get("evalAtZero"):\n            evaluate(0)\n        model.train(); reset_peak(); tstart = time.time(); nonfinite = 0\n        for s, bidx in enumerate(batches):\n            t1 = time.time()\n            chunk = [train[i] for i in bidx]\n            while True:\n                try:\n                    opt.zero_grad(set_to_none=True); tl = ta = tm = 0.0\n                    for k in range(0, len(chunk), micro):\n                        sub = chunk[k:k + micro]\n                        seqs = [p["c"] for p in sub] + [p["r"] for p in sub]\n                        L = max(len(x) for x in seqs)\n                        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); att = torch.zeros_like(ids)\n                        for q, x in enumerate(seqs):\n                            ids[q, :len(x)] = torch.tensor(x); att[q, :len(x)] = 1\n                        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):\n                            rew = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV)).logits.squeeze(-1).float()\n                        rc, rr = rew[:len(sub)], rew[len(sub):]\n                        loss = -torch.nn.functional.logsigmoid(rc - rr).mean() * (len(sub) / len(chunk))\n                        scaler.scale(loss).backward()\n                        tl += loss.item(); ta += (rc > rr).float().sum().item(); tm += (rc - rr).sum().item()\n                    break\n                except torch.OutOfMemoryError:\n                    R.setdefault("oomRetries", []).append({"step": s + 1, "micro": micro})\n                    opt.zero_grad(set_to_none=True); free()\n                    if micro == 1:\n                        raise\n                    micro = max(1, micro // 2)\n            scaler.unscale_(opt)\n            gn = torch.nn.utils.clip_grad_norm_(params, 1.0).item()\n            if not math.isfinite(gn):\n                nonfinite += 1\n            scaler.step(opt); scaler.update(); sched.step()\n            sync()\n            metrics.append({"step": s + 1, "pairs": (s + 1) * B, "trainLoss": round(tl, 5), "trainAcc": round(ta / len(chunk), 4),\n                            "margin": round(tm / len(chunk), 4), "gradNorm": round(gn, 4) if math.isfinite(gn) else None,\n                            "lr": sched.get_last_lr()[0], "sec": round(time.time() - t1, 3), "maxLen": max(max(len(p["c"]), len(p["r"])) for p in chunk),\n                            "scale": scaler.get_scale()})\n            if (s + 1) % 25 == 0 or s == 0:\n                print(json.dumps(metrics[-1]), "elapsed", round(time.time() - tstart), flush=True)\n            if (s + 1) in evals_at:\n                R["trainSecSoFar"] = round(time.time() - tstart, 1)\n                evaluate(s + 1)\n            if job.get("maxMinutes") and time.time() - tstart > 60 * job["maxMinutes"]:\n                R["stoppedEarlyAtStep"] = s + 1; break\n        R["metrics"] = metrics\n        R["trainSec"] = round(sum(m["sec"] for m in metrics), 1)\n        R["sPerStepMedian"] = float(np.median([m["sec"] for m in metrics[2:]])) if len(metrics) > 3 else None\n        R["peakMemGB"] = peak(); R["nonFiniteGradSteps"] = nonfinite\n        if job.get("rb2PerSubset"):\n            model.eval()\n            items = rb2_subset(job["rb2PerSubset"]); flat = []\n            for it in items:\n                for resp in it["chosen"][:1] + it["rejected"][:3]:\n                    flat.append(ids_of(tok, it["prompt"], resp)[:job.get("rb2MaxTokens", 2048)])\n            sc = score_seqs(model, flat, tok.pad_token_id)\n            res = {}\n            for k, it in enumerate(items):\n                v = sc[4 * k:4 * k + 4]\n                res.setdefault(it["subset"], []).append(float(v[0] > max(v[1:])))\n            R["rb2"] = {sub: {"n": len(v), "acc": float(np.mean(v))} for sub, v in res.items()}\n            R["rb2Mean"] = float(np.mean([v["acc"] for v in R["rb2"].values()]))\n            json.dump({"proxy": tag, "ids": [it["id"] for it in items], "subset": [it["subset"] for it in items], "scores": sc},\n                      open(os.path.join(WORK, "proxy_rb2_%s.json" % tag), "w"))\n        if job.get("saveDir"):\n            d = os.path.join(WORK, job["saveDir"])\n            model.config.use_cache = True\n            model.half().save_pretrained(d, safe_serialization=True)\n            tok.save_pretrained(d)\n            R["saved"] = sorted(os.listdir(d))\n        R["ok"] = True\n\n    elif kind == "trlcheck":\n        from datasets import Dataset\n        from trl import RewardConfig, RewardTrainer\n        import trl\n        R["trl"] = trl.__version__\n        tok = rm_tokenizer()\n        rows, _ = select_pairs(tok, "train_prefs", job["pairs"], seed=0)\n        from datasets import load_dataset\n        tr = load_dataset(UF, split="train_prefs").shuffle(seed=0)\n        keep = set(p["pid"] for p in rows); recs = []\n        for ex in tr:\n            if ex["prompt_id"] in keep and ex["chosen"][-1]["content"] != ex["rejected"][-1]["content"]:\n                recs.append({"chosen": conv(ex["prompt"], ex["chosen"][-1]["content"]),\n                             "rejected": conv(ex["prompt"], ex["rejected"][-1]["content"])})\n            if len(recs) >= len(rows):\n                break\n        torch.manual_seed(0)\n        model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1, dtype=torch.float32, device_map=DEV)\n        cfg = RewardConfig(output_dir=os.path.join(WORK, "trl_rm"), max_length=MAXLEN, per_device_train_batch_size=job["batchPairs"],\n                           gradient_accumulation_steps=1, max_steps=job["steps"], learning_rate=job["lr"], fp16=True, bf16=False,\n                           lr_scheduler_type="cosine", warmup_steps=0.03,\n                           gradient_checkpointing=True, report_to="none", logging_steps=1, save_strategy="no", seed=0, data_seed=0)\n        trainer = RewardTrainer(model=model, args=cfg, train_dataset=Dataset.from_list(recs), processing_class=tok)\n        R["trainableDtypes"] = sorted({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})\n        R["trlKeptPairs"] = len(trainer.train_dataset); R["pairsGiven"] = len(recs)\n        own = [(ids_of(tok, r["chosen"][0]["content"], r["chosen"][1]["content"]), ids_of(tok, r["rejected"][0]["content"], r["rejected"][1]["content"])) for r in recs]\n        R["idsEqual"] = all(list(trainer.train_dataset[i]["chosen_ids"]) == own[i][0] and list(trainer.train_dataset[i]["rejected_ids"]) == own[i][1]\n                            for i in range(len(trainer.train_dataset)))\n        batch = next(iter(trainer.get_train_dataloader()))\n        batch = {k: v.to(DEV) for k, v in batch.items()}\n        trainer.model.eval()\n        with torch.no_grad():\n            lib = trainer.compute_loss(trainer.model, dict(batch)).item()\n            h = trainer.model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state\n            last = batch["attention_mask"].sum(-1) - 1                       # right padding: the last real token\n            r = trainer.model.score(h[torch.arange(h.shape[0]), last]).squeeze(-1).float()\n            n = r.shape[0] // 2\n            ownl = -torch.nn.functional.logsigmoid(r[:n] - r[n:]).mean().item()\n        R["lossCheck"] = {"batchShape": list(batch["input_ids"].shape), "trl": lib, "own": ownl, "absDiff": abs(lib - ownl)}\n        print(R["lossCheck"], flush=True)\n        trainer.model.train(); reset_peak(); t = time.time()\n        out = trainer.train()\n        R["trainSec"] = round(time.time() - t, 1); R["peakMemGB"] = peak()\n        R["log"] = [h for h in trainer.state.log_history if "loss" in h]\n        R["ok"] = True\n\n    save()\nexcept Exception as e:\n    R["ok"] = False; R["error"] = repr(e)[:2000]; R["traceback"] = traceback.format_exc()[-4000:]\n    save(); print(R["traceback"], flush=True)\n'
open("/kaggle/working/worker_rm.py", "w").write(WORKER)
import torch, transformers, trl, accelerate, datasets
ALL = {"date": time.strftime("%Y-%m-%d"), "gpu": torch.cuda.get_device_name(0), "gpuCount": torch.cuda.device_count(),
       "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
               "trl": trl.__version__, "accelerate": accelerate.__version__, "datasets": datasets.__version__},
       "runs": []}
print(ALL, flush=True)
T0 = time.time()
POOL = (glob.glob("/kaggle/input/**/pool.json", recursive=True) or ["/kaggle/working/pool.json"])[0]
print("pool file", POOL, os.path.exists(POOL))
def save():
    json.dump(ALL, open("/kaggle/working/llt_rm_%s.json" % TAG, "w"), indent=1)
def job(tag, kind, cap=3600, **kw):
    jf = "/kaggle/working/job_%s.json" % tag; rf = "/kaggle/working/res_%s.json" % tag
    json.dump(dict(tag=tag, kind=kind, pool=POOL, **kw), open(jf, "w"))
    t = time.time()
    try:
        p = subprocess.run([sys.executable, "/kaggle/working/worker_rm.py", jf, rf], capture_output=True, text=True, timeout=cap)
        tail = (p.stdout or "")[-3000:] + (p.stderr or "")[-1500:]
    except subprocess.TimeoutExpired:
        tail = "TIMEOUT after %ss" % cap
    try:
        r = json.load(open(rf))
    except Exception:
        r = {"tag": tag, "ok": False, "error": "no result file / crashed / timeout"}
    r["wallSec"] = round(time.time() - t, 1); r["tail"] = tail
    ALL["runs"].append(r); save()
    print(tag, {k: r.get(k) for k in ("ok", "sPerStepMedian", "peakMemGB", "error", "totalGenSec", "rb2Mean", "lossCheck")}, flush=True)
    print(tail[-1500:], flush=True)
    print("elapsed min", round((time.time() - T0) / 60, 1), flush=True)


In [ ]:

# Part 1. Best-of-n pool: 64 held-out UltraFeedback prompts x 64 samples from SmolLM2-360M-Instruct (fp32 sampling,
# temperature 1), then the gold reward model on the pool, on 1,000 held-out pairs and on a RewardBench 2 subset.
job("pool-sample", "sample", 4200, prompts=64, n=64, promptsPerBatch=2, newTokens=256, maxRefTokens=160, seed=0)
job("pool-gold", "gold", 2400, heldPairs=1000, rb2PerSubset=60)
# Part 2. Own Bradley-Terry loss vs RewardTrainer.compute_loss on one batch, then a short RewardTrainer run.
job("trl-check", "trlcheck", 900, pairs=64, batchPairs=4, steps=40, lr=1e-5)
# Part 3. Proxy reward models: SmolLM2-360M + scalar head, full fine-tune, 3,000 UltraFeedback pairs, 1 epoch.
common = dict(pairs=3000, heldPairs=1000, batchPairs=8, scorePool=True, rb2PerSubset=60)
job("rm-s0", "train", 4200, seed=0, lr=1e-5, evalFractions=[0.25, 0.5, 1.0], evalAtZero=True, saveDir="rm-s0", **common)
job("rm-s1", "train", 3000, seed=1, lr=1e-5, evalFractions=[1.0], saveDir="rm-s1", **common)
job("rm-bal-s0", "train", 3000, seed=0, lr=1e-5, balanced=True, evalFractions=[1.0], saveDir="rm-bal-s0", **common)
# Part 4. The RewardConfig default learning rate, 1e-4, on a full fine-tune (800 pairs, 100 steps).
job("rm-lr1e-4", "train", 1200, seed=0, lr=1e-4, pairs=800, heldPairs=1000, batchPairs=8, evalFractions=[1.0])
ALL["totalMinutes"] = round((time.time() - T0) / 60, 1); save(); print("DONE", ALL["totalMinutes"])
